In [2]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/raw")

print("Data path:", DATA_PATH.resolve())
print("Exists:", DATA_PATH.exists())

Data path: C:\Users\Fayaz\Retail-Business-Intelligence\data\raw
Exists: True


In [3]:
files = [
    "begin_inventory.csv",
    "end_inventory.csv",
    "purchase_prices.csv",
    "purchases.csv",
    "sales.csv",
    "vendor_invoice.csv"
]

for file in files:
    path = DATA_PATH / file
    print(file, "->", path.exists())

begin_inventory.csv -> True
end_inventory.csv -> True
purchase_prices.csv -> True
purchases.csv -> True
sales.csv -> True
vendor_invoice.csv -> True


In [4]:
begin_inventory = pd.read_csv(DATA_PATH / "begin_inventory.csv")
end_inventory = pd.read_csv(DATA_PATH / "end_inventory.csv")
purchase_prices = pd.read_csv(DATA_PATH / "purchase_prices.csv")
purchases = pd.read_csv(DATA_PATH / "purchases.csv")
sales = pd.read_csv(DATA_PATH / "sales.csv")
vendor_invoice = pd.read_csv(DATA_PATH / "vendor_invoice.csv")

In [6]:
datasets = {
    "begin_inventory": begin_inventory,
    "end_inventory": end_inventory,
    "purchase_prices": purchase_prices,
    "purchases": purchases,
    "sales": sales,
    "vendor_invoice": vendor_invoice
}

In [7]:
for name, df in datasets.items():

    print("=" * 70)
    print(f"DATASET: {name}")
    print("=" * 70)

    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nDuplicate rows:")
    print(df.duplicated().sum())

    print("\nData types:")
    print(df.dtypes)

    print()

DATASET: begin_inventory
Shape: (206529, 9)

Columns:
['InventoryId', 'Store', 'City', 'Brand', 'Description', 'Size', 'onHand', 'Price', 'startDate']

Missing values:
InventoryId    0
Store          0
City           0
Brand          0
Description    0
Size           0
onHand         0
Price          0
startDate      0
dtype: int64

Duplicate rows:
0

Data types:
InventoryId        str
Store            int64
City               str
Brand            int64
Description        str
Size               str
onHand           int64
Price          float64
startDate          str
dtype: object

DATASET: end_inventory
Shape: (224489, 9)

Columns:
['InventoryId', 'Store', 'City', 'Brand', 'Description', 'Size', 'onHand', 'Price', 'endDate']

Missing values:
InventoryId       0
Store             0
City           1284
Brand             0
Description       0
Size              0
onHand            0
Price             0
endDate           0
dtype: int64

Duplicate rows:
0

Data types:
InventoryId        str


In [8]:
date_columns = {
    "begin_inventory": ["startDate"],
    "end_inventory": ["endDate"],
    "purchases": ["PODate", "ReceivingDate", "InvoiceDate", "PayDate"],
    "sales": ["SalesDate"],
    "vendor_invoice": ["InvoiceDate", "PODate", "PayDate"]
}

for dataset_name, columns in date_columns.items():

    df = datasets[dataset_name]

    print(f"\n{dataset_name}")

    for column in columns:
        converted = pd.to_datetime(df[column], errors="coerce")

        print(
            column,
            "invalid dates:",
            converted.isna().sum(),
            "min:",
            converted.min(),
            "max:",
            converted.max()
        )


begin_inventory
startDate invalid dates: 0 min: 2024-01-01 00:00:00 max: 2024-01-01 00:00:00

end_inventory
endDate invalid dates: 0 min: 2024-12-31 00:00:00 max: 2024-12-31 00:00:00

purchases
PODate invalid dates: 0 min: 2023-12-20 00:00:00 max: 2024-12-23 00:00:00
ReceivingDate invalid dates: 0 min: 2024-01-01 00:00:00 max: 2024-12-31 00:00:00
InvoiceDate invalid dates: 0 min: 2024-01-04 00:00:00 max: 2025-01-10 00:00:00
PayDate invalid dates: 0 min: 2024-02-04 00:00:00 max: 2025-02-19 00:00:00

sales
SalesDate invalid dates: 0 min: 2024-01-01 00:00:00 max: 2024-12-31 00:00:00

vendor_invoice
InvoiceDate invalid dates: 0 min: 2024-01-04 00:00:00 max: 2025-01-10 00:00:00
PODate invalid dates: 0 min: 2023-12-20 00:00:00 max: 2024-12-23 00:00:00
PayDate invalid dates: 0 min: 2024-02-04 00:00:00 max: 2025-02-19 00:00:00


Relationships:

In [9]:
store_city = (
    begin_inventory
    .groupby("Store")["City"]
    .nunique(dropna=True)
)

print(store_city.sort_values(ascending=False).head(20))

Store
1     1
2     1
3     1
4     1
5     1
6     1
7     1
8     1
9     1
10    1
11    1
12    1
13    1
14    1
15    1
16    1
17    1
18    1
19    1
20    1
Name: City, dtype: int64


In [10]:
print("Stores with multiple cities:")
print(store_city[store_city > 1])

Stores with multiple cities:
Series([], Name: City, dtype: int64)


In [11]:
end_store_city = (
    end_inventory
    .groupby("Store")["City"]
    .nunique(dropna=True)
)

print("Stores with multiple cities:")
print(end_store_city[end_store_city > 1])

Stores with multiple cities:
Series([], Name: City, dtype: int64)


In [12]:
print("Missing City rows:", end_inventory["City"].isna().sum())

Missing City rows: 1284


In [13]:
vendor_mapping = (
    purchases
    .groupby("VendorNumber")["VendorName"]
    .nunique()
    .sort_values(ascending=False)
)

print(vendor_mapping[vendor_mapping > 1])

VendorNumber
2000    2
1587    2
4425    2
Name: VendorName, dtype: int64


In [14]:
vendor_conflicts = (
    purchases
    .groupby("VendorNumber")["VendorName"]
    .unique()
)

vendor_conflicts = vendor_conflicts[
    vendor_conflicts.apply(len) > 1
]

print(vendor_conflicts)

VendorNumber
1587    [VINEYARD BRANDS INC        , VINEYARD BRANDS ...
2000    [SOUTHERN WINE & SPIRITS NE , SOUTHERN GLAZERS...
4425      [MARTIGNETTI COMPANIES , MARTIGNETTI COMPANIES]
Name: VendorName, dtype: object


In [15]:
print("Number of conflicting vendor numbers:",
      len(vendor_conflicts))

Number of conflicting vendor numbers: 3


In [17]:
purchases_vendor_map = (
    purchases.groupby("VendorNumber")["VendorName"].nunique()
)

In [18]:
sales_vendor_map = (
    sales.groupby("VendorNo")["VendorName"].nunique()
)

In [20]:
price_vendor_map = (
    purchase_prices.groupby("VendorNumber")["VendorName"].nunique()
)

In [21]:
invoice_vendor_map = (
    vendor_invoice.groupby("VendorNumber")["VendorName"].nunique()
)

In [22]:
print("Purchases conflicts:")
print(purchases_vendor_map[purchases_vendor_map > 1])

print("\nSales conflicts:")
print(sales_vendor_map[sales_vendor_map > 1])

print("\nPurchase price conflicts:")
print(price_vendor_map[price_vendor_map > 1])

print("\nInvoice conflicts:")
print(invoice_vendor_map[invoice_vendor_map > 1])

Purchases conflicts:
VendorNumber
1587    2
2000    2
4425    2
Name: VendorName, dtype: int64

Sales conflicts:
VendorNo
1587    2
2000    2
4425    2
Name: VendorName, dtype: int64

Purchase price conflicts:
VendorNumber
1587    2
1703    3
2000    2
4425    2
Name: VendorName, dtype: int64

Invoice conflicts:
VendorNumber
1587    2
2000    2
4425    2
Name: VendorName, dtype: int64


In [26]:
# Check the actual conflicting vendor names

print("=== PURCHASES ===")
print(
    purchases[
        purchases["VendorNumber"].isin([1587, 1703, 2000, 4425])
    ][["VendorNumber", "VendorName"]]
    .drop_duplicates()
    .sort_values(["VendorNumber", "VendorName"])
)

print("\n=== SALES ===")
print(
    sales[
        sales["VendorNo"].isin([1587, 1703, 2000, 4425])
    ][["VendorNo", "VendorName"]]
    .drop_duplicates()
    .sort_values(["VendorNo", "VendorName"])
)

print("\n=== PURCHASE PRICES ===")
print(
    purchase_prices[
        purchase_prices["VendorNumber"].isin([1587, 1703, 2000, 4425])
    ][["VendorNumber", "VendorName"]]
    .drop_duplicates()
    .sort_values(["VendorNumber", "VendorName"])
)

=== PURCHASES ===
         VendorNumber                       VendorName
7582             1587      VINEYARD BRANDS INC        
2100628          1587      VINEYARD BRANDS LLC        
116026           1703  ALISA CARR BEVERAGES           
1593834          2000      SOUTHERN GLAZERS W&S OF NE 
6628             2000      SOUTHERN WINE & SPIRITS NE 
4297             4425            MARTIGNETTI COMPANIES
4295             4425           MARTIGNETTI COMPANIES 

=== SALES ===
          VendorNo                       VendorName
1009          1587      VINEYARD BRANDS INC        
10748135      1587      VINEYARD BRANDS LLC        
302447        1703  ALISA CARR BEVERAGES           
8225794       2000      SOUTHERN GLAZERS W&S OF NE 
8             2000      SOUTHERN WINE & SPIRITS NE 
40            4425            MARTIGNETTI COMPANIES
2555          4425           MARTIGNETTI COMPANIES 

=== PURCHASE PRICES ===
       VendorNumber                           VendorName
1520           1587          

In [24]:
print("\n=== VENDOR INVOICE ===")
print(
    vendor_invoice[
        vendor_invoice["VendorNumber"].isin([1587, 1703, 2000, 4425])
    ][["VendorNumber", "VendorName"]]
    .drop_duplicates()
    .sort_values(["VendorNumber", "VendorName"])
)


=== VENDOR INVOICE ===
      VendorNumber                       VendorName
81            1587      VINEYARD BRANDS INC        
5006          1587      VINEYARD BRANDS LLC        
379           1703  ALISA CARR BEVERAGES           
3793          2000      SOUTHERN GLAZERS W&S OF NE 
67            2000      SOUTHERN WINE & SPIRITS NE 
43            4425            MARTIGNETTI COMPANIES
42            4425           MARTIGNETTI COMPANIES 


In [27]:
product_test = (
    purchase_prices
    .groupby(["Brand", "Description", "Size"])
    .size()
)

print("Duplicate product combinations:",
      (product_test > 1).sum())

Duplicate product combinations: 0


In [28]:
brand_description = (
    purchase_prices
    .groupby("Brand")["Description"]
    .nunique()
)

print(
    "Brands with multiple descriptions:",
    (brand_description > 1).sum()
)

Brands with multiple descriptions: 0


In [29]:
brand_size = (
    purchase_prices
    .groupby("Brand")["Size"]
    .nunique()
)

print(
    "Brands with multiple sizes:",
    (brand_size > 1).sum()
)

Brands with multiple sizes: 0


In [30]:
brand_vendor = (
    purchase_prices
    .groupby("Brand")["VendorNumber"]
    .nunique()
)

print(
    "Brands with multiple vendors:",
    (brand_vendor > 1).sum()
)

Brands with multiple vendors: 0


In [31]:
# ==========================================
# RELATIONSHIP 4 — INVENTORY ID
# ==========================================

print("=== DUPLICATE CHECK ===")

print("Begin InventoryId duplicates:",
      begin_inventory["InventoryId"].duplicated().sum())

print("End InventoryId duplicates:",
      end_inventory["InventoryId"].duplicated().sum())

print("Purchase InventoryId duplicates:",
      purchases["InventoryId"].duplicated().sum())

print("Sales InventoryId duplicates:",
      sales["InventoryId"].duplicated().sum())


# ==========================================
# BEGIN vs END INVENTORY
# ==========================================

print("\n=== BEGIN vs END INVENTORY ===")

begin_ids = set(begin_inventory["InventoryId"])
end_ids = set(end_inventory["InventoryId"])

print("Begin Inventory IDs:", len(begin_ids))
print("End Inventory IDs:", len(end_ids))
print("Common IDs:", len(begin_ids & end_ids))
print("Only in Begin:", len(begin_ids - end_ids))
print("Only in End:", len(end_ids - begin_ids))


# ==========================================
# SALES vs INVENTORY
# ==========================================

print("\n=== SALES vs INVENTORY ===")

sales_ids = set(sales["InventoryId"])

print("Begin IDs found in Sales:",
      len(begin_ids & sales_ids))

print("End IDs found in Sales:",
      len(end_ids & sales_ids))

print("Sales IDs not found in Begin:",
      len(sales_ids - begin_ids))

print("Sales IDs not found in End:",
      len(sales_ids - end_ids))


# ==========================================
# PURCHASES vs INVENTORY
# ==========================================

print("\n=== PURCHASES vs INVENTORY ===")

purchase_ids = set(purchases["InventoryId"])

print("Begin IDs found in Purchases:",
      len(begin_ids & purchase_ids))

print("End IDs found in Purchases:",
      len(end_ids & purchase_ids))

print("Purchase IDs not found in Begin:",
      len(purchase_ids - begin_ids))

print("Purchase IDs not found in End:",
      len(purchase_ids - end_ids))

=== DUPLICATE CHECK ===
Begin InventoryId duplicates: 0
End InventoryId duplicates: 0
Purchase InventoryId duplicates: 2126567
Sales InventoryId duplicates: 12557811

=== BEGIN vs END INVENTORY ===
Begin Inventory IDs: 206529
End Inventory IDs: 224489
Common IDs: 174976
Only in Begin: 31553
Only in End: 49513

=== SALES vs INVENTORY ===
Begin IDs found in Sales: 203508
End IDs found in Sales: 217184
Sales IDs not found in Begin: 64044
Sales IDs not found in End: 50368

=== PURCHASES vs INVENTORY ===
Begin IDs found in Purchases: 177035
End IDs found in Purchases: 213001
Purchase IDs not found in Begin: 68872
Purchase IDs not found in End: 32906


In [33]:
# ==========================================
# RELATIONSHIP 5 — PONUMBER
# ==========================================

print("=== DUPLICATES ===")

print(
    "Duplicate PO numbers in purchases:",
    purchases["PONumber"].duplicated().sum()
)

print(
    "Duplicate PO numbers in vendor invoices:",
    vendor_invoice["PONumber"].duplicated().sum()
)


# ==========================================
# PO OVERLAP
# ==========================================

print("\n=== PO OVERLAP ===")

purchase_pos = set(purchases["PONumber"])
invoice_pos = set(vendor_invoice["PONumber"])

print("Purchase POs:", len(purchase_pos))
print("Invoice POs:", len(invoice_pos))

print("Common POs:",
      len(purchase_pos & invoice_pos))

print("POs in purchases but not invoices:",
      len(purchase_pos - invoice_pos))

print("POs in invoices but not purchases:",
      len(invoice_pos - purchase_pos))

=== DUPLICATES ===
Duplicate PO numbers in purchases: 2366931
Duplicate PO numbers in vendor invoices: 0

=== PO OVERLAP ===
Purchase POs: 5543
Invoice POs: 5543
Common POs: 5543
POs in purchases but not invoices: 0
POs in invoices but not purchases: 0


In [34]:
# ==========================================
# RELATIONSHIP 6 — PURCHASE PRICE MATCHING
# ==========================================

price_keys = set(
    zip(
        purchase_prices["Brand"],
        purchase_prices["VendorNumber"]
    )
)

purchase_keys = set(
    zip(
        purchases["Brand"],
        purchases["VendorNumber"]
    )
)

matched_keys = purchase_keys & price_keys
missing_keys = purchase_keys - price_keys

print("=== PURCHASE PRICE MATCHING ===")

print(
    "Unique Brand + Vendor combinations in purchases:",
    len(purchase_keys)
)

print(
    "Unique Brand + Vendor combinations in purchase prices:",
    len(price_keys)
)

print(
    "Purchase combinations found in price table:",
    len(matched_keys)
)

print(
    "Purchase combinations missing from price table:",
    len(missing_keys)
)

print(
    "Matching percentage:",
    round(
        len(matched_keys) / len(purchase_keys) * 100,
        2
    ),
    "%"
)

=== PURCHASE PRICE MATCHING ===
Unique Brand + Vendor combinations in purchases: 10693
Unique Brand + Vendor combinations in purchase prices: 12261
Purchase combinations found in price table: 10649
Purchase combinations missing from price table: 44
Matching percentage: 99.59 %


In [35]:
print(store_city_begin[store_city_begin > 1])
print((store_city_begin > 1).sum())

print(store_city_end[store_city_end > 1])
print((store_city_end > 1).sum())

print(end_inventory["City"].isna().sum())

NameError: name 'store_city_begin' is not defined